In [ ]:
# IMPORT THE LIBRARIES

import re
from pathlib import Path
import matplotlib.pyplot as plt

In [ ]:
# SET STYLE

plt.style.use("ggplot")
plt.rcParams.update({
    "figure.figsize": (12, 6),
    "font.size": 18,
    "axes.titlesize": 24,
    "axes.labelsize": 22,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "axes.labelcolor": "black",
    "xtick.color": "black",
    "ytick.color": "black",
    "text.color": "black",
    "font.weight": "bold",
    "text.usetex": True,
    "mathtext.fontset": "dejavusans",
    "font.family": "sans-serif",
})

In [ ]:
# SCRAPE THE OUTPUT AND SEQUENTIAL DIRECTORY AND EXTRACT RESULTS

roots = [
    Path("OUT") / "test1",
    Path("OUT") / "seq",
]

records = []

for root in roots:
    for file in root.iterdir():
        if not file.is_file():
            continue

        if not file.name.endswith(".out"): 
            continue

        text = file.read_text()

        # parse header block
        header = {}
        for line in text.splitlines():
            if line.strip() == "---":
                continue
            if re.match(r"\s*---\s*$", line):
                continue
            if line.startswith(">>>>"):
                break
            m = re.match(r"(\w+):\s*(.*)", line)
            if m:
                header[m.group(1)] = m.group(2).strip()

        # required indexing fields
        try:
            partition = header["partition"]
            nprocs = int(header["nprocs"])
            num_neurons = int(header["num_neurons"])
            num_epochs = int(header["num_epochs"])
        except KeyError:
            print(f"missing header in {file.name}")
            continue

        # parse executions
        blocks = re.split(r">>>>>>>>>>>>\s+\d+", text)
        execs = []
        for block in blocks[1:]:
            m1 = re.search(r"RIGHT\s+(\d+)", block)
            m2 = re.search(r"TRAIN\s+([\d.]+)", block)
            m3 = re.search(r"TEST\s+([\d.]+)", block)
            m4 = re.search(r"TOTAL\s+([\d.]+)", block)
            if m1 and m2 and m3 and m4:
                RIGHT = int(m1.group(1))
                TRAIN = float(m2.group(1))
                TEST = float(m3.group(1))
                TOTAL = float(m4.group(1))
                execs.append((RIGHT, TRAIN, TEST, TOTAL))

        if not execs:
            print(f"no valid execs in {file.name}")
            continue

        # choose minimal TOTAL
        best = min(execs, key=lambda x: x[3])
        RIGHT, TRAIN, TEST, TOTAL = best

        record = {
            "partition": partition,
            "nprocs": nprocs,
            "num_neurons": num_neurons,
            "num_epochs": num_epochs,
            "RIGHT": RIGHT,
            "TOTAL": TOTAL,
            "TRAIN_TEST_RATIO": TRAIN/TEST if TEST > 0 else None,
            "filename": file.name
        }
        records.append(record)

In [ ]:
# SCRAPE THE SEQUENTIAL OUTPUT DIRECTORY AND EXTRACT RESULTS

root_seq = Path("OUT") / "seq"


for file in root_seq.iterdir():
    if not file.is_file():
        continue

    if not file.name.endswith(".out"): 
        continue

    text = file.read_text()

    # parse header block
    header = {}
    for line in text.splitlines():
        if line.strip() == "---":
            continue
        if re.match(r"\s*---\s*$", line):
            continue
        if line.startswith(">>>>"):
            break
        m = re.match(r"(\w+):\s*(.*)", line)
        if m:
            header[m.group(1)] = m.group(2).strip()

    # required indexing fields
    try:
        partition = header["partition"]
        nprocs = int(header["nprocs"])
        num_neurons = int(header["num_neurons"])
        num_epochs = int(header["num_epochs"])
    except KeyError:
        print(f"missing header in {file.name}")
        continue

    # parse executions
    blocks = re.split(r">>>>>>>>>>>>\s+\d+", text)
    execs = []
    for block in blocks[1:]:
        m1 = re.search(r"RIGHT\s+(\d+)", block)
        m2 = re.search(r"TRAIN\s+([\d.]+)", block)
        m3 = re.search(r"TEST\s+([\d.]+)", block)
        m4 = re.search(r"TOTAL\s+([\d.]+)", block)
        if m1 and m2 and m3 and m4:
            RIGHT = int(m1.group(1))
            TRAIN = float(m2.group(1))
            TEST = float(m3.group(1))
            TOTAL = float(m4.group(1))
            execs.append((RIGHT, TRAIN, TEST, TOTAL))

    if not execs:
        print(f"no valid execs in {file.name}")
        continue

    # choose minimal TOTAL
    best = min(execs, key=lambda x: x[3])
    RIGHT, TRAIN, TEST, TOTAL = best

    record = {
        "partition": partition,
        "nprocs": nprocs,
        "num_neurons": num_neurons,
        "num_epochs": num_epochs,
        "RIGHT": RIGHT,
        "TOTAL": TOTAL,
        "TRAIN_TEST_RATIO": TRAIN/TEST if TEST > 0 else None,
        "filename": file.name
    }
    records.append(record)

In [ ]:
# CONVERT TO A NICE INDEXED DICTIONARY

indexed = {}

for rec in records:
    key = (
        rec["partition"],
        rec["nprocs"],
        rec["num_neurons"],
        rec["num_epochs"],
    )
    indexed[key] = {
        "RIGHT": rec["RIGHT"],
        "TOTAL": rec["TOTAL"],
        "TRAIN_TEST_RATIO": rec["TRAIN_TEST_RATIO"],
        "filename": rec["filename"],
    }

for k, v in indexed.items():
    print(k, v)

In [ ]:
# COMPUTE THE METRICS

indexed_metrics = {}

for (partition, nprocs, num_neurons, num_epochs), par in indexed.items():
    key_seq = (partition, 1, num_neurons, num_epochs)
    seq = indexed[key_seq]

    # Extract raw fields
    T_par = par["TOTAL"]
    T_seq = seq["TOTAL"]

    RIGHT_par = par["RIGHT"]
    RIGHT_seq = seq["RIGHT"]

    RATIO_par = par["TRAIN_TEST_RATIO"]
    RATIO_seq = seq["TRAIN_TEST_RATIO"]

    # Derived metrics
    speedup = T_seq / T_par if T_par > 0 else float("nan")
    efficiency = speedup / nprocs if nprocs > 0 else float("nan")
    right_ratio = RIGHT_par / RIGHT_seq if RIGHT_seq > 0 else float("nan")
    ratio_ratio = RATIO_par / RATIO_seq if RATIO_seq > 0 else float("nan")

    indexed_metrics[(partition, nprocs, num_neurons, num_epochs)] = {
        "RIGHT": RIGHT_par,
        "TOTAL": T_par,
        "TRAIN_TEST_RATIO": RATIO_par,
        "speedup": speedup,
        "efficiency": efficiency,
        "right_ratio": right_ratio,
        "ratio_ratio": ratio_ratio
    }

In [ ]:
# GRAPHS

partitions = set(k[0] for k in indexed_metrics.keys())
neurons_list = [135, 250]
epochs_list = [10, 100]

for partition in partitions:
    fig, axes = plt.subplots(2, 2, figsize=(20, 12))
    fig.suptitle(f"Partition: {partition}", fontsize=30)

    for i, num_neurons in enumerate(neurons_list):
        for j, num_epochs in enumerate(epochs_list):
            ax = axes[i, j]

            # collect x and y
            nprocs_vals = []
            total_times = []
            right_ratios = []

            for key, data in indexed_metrics.items():
                p, nprocs, neurons, epochs = key
                if p == partition and neurons == num_neurons and epochs == num_epochs:
                    if nprocs == 1:  # skip sequential baseline
                        continue
                    nprocs_vals.append(nprocs)
                    total_times.append(data['TOTAL'])
                    right_ratios.append(data['RIGHT'])

            if not nprocs_vals:
                ax.set_title(f"Neurones={num_neurons}, Epochs={num_epochs} (no data)")
                continue

            # sort by nprocs
            sorted_data = sorted(zip(nprocs_vals, total_times, right_ratios))
            nprocs_vals, total_times, right_ratios = zip(*sorted_data)

            # plot TOTAL time on primary y-axis
            ax.plot(nprocs_vals, total_times, marker='o', label='temps')
            ax.set_xlabel("nprocs")
            ax.set_ylabel("temps")
            ax.set_ylim(bottom=0)

            seq_key = (partition, 1, num_neurons, num_epochs)
            seq_right = indexed_metrics.get(seq_key, {}).get('RIGHT', None)

            # create secondary y-axis for RIGHT ratio
            ax2 = ax.twinx()
            ax2.plot(nprocs_vals, right_ratios, marker='x', linestyle='--', color='red', alpha=0.5, label='encerts')
            ax2.set_ylabel("encerts")
            ax2.set_ylim(bottom=0, top=1.1 * max(right_ratios))
            ax2.grid(False)
            ax2.axhline(y=seq_right, color='red', linestyle=':', linewidth=1.5, label='encerts (seq)')

            ax.set_title(f"Neurones={num_neurons}, Epochs={num_epochs}")
            # merge legends
            lines, labels = ax.get_legend_handles_labels()
            lines2, labels2 = ax2.get_legend_handles_labels()
            ax2.legend(lines + lines2, labels + labels2, loc="center right")

    plt.tight_layout(rect=[0, 0, 1, 0.96])
    plt.savefig(f"{partition}.pdf")  # pdf for small size
    plt.show()

In [ ]:
# TABLES

partitions = set(k[0] for k in indexed_metrics.keys())
neurons_list = [135, 250]
epochs_list = [10, 100]

latex_tables = []

for partition in partitions:
    for num_neurons in neurons_list:
        for num_epochs in epochs_list:
            # collect rows
            rows = []
            seq_key = (partition, 1, num_neurons, num_epochs)  # sequential baseline nprocs=1
            seq_data = indexed_metrics.get(seq_key)
            if seq_data is None:
                print(f"No sequential data for {partition}, neurons={num_neurons}, epochs={num_epochs}")
                continue

            for key, data in indexed_metrics.items():
                p, nprocs, neurons, epochs = key
                if p == partition and neurons == num_neurons and epochs == num_epochs:
                    total = data['TOTAL']
                    right = data['RIGHT']
                    speedup = seq_data['TOTAL'] / total
                    efficiency = speedup / nprocs
                    rows.append((nprocs, total, right, speedup, efficiency))

            if not rows:
                continue

            # sort by nprocs
            rows.sort(key=lambda x: x[0])

            # generate LaTeX
            table = "\\begin{table}[H]\n\\centering\n"
            table += "\\begin{tabular}{|c|c|c|c|c|}\n\\hline\n"
            table += "Nº Processos & Temps & Encerts & Speedup & Eficiència \\\\\n\\hline\n"

            for row in rows:
                nprocs, total, right, speedup, efficiency = row
                table += f"{nprocs} & {total:.4f} & {int(right)} & {speedup:.4f} & {efficiency:.4f} \\\\\n"

            table += f"\\hline\n\\end{{tabular}}\n\\caption{{Partició \\texttt{{{partition}}}, {num_neurons} neurones, {num_epochs} èpoques}}\n\\end{{table}}\n"

            latex_tables.append(table)

# output all tables
for t in latex_tables:
    print(t)

In [ ]:
# SUMMARY TABLE BEST TIME

partitions = set(k[0] for k in indexed_metrics.keys())
neurons_list = [135, 250]
epochs_list = [10, 100]

summary_rows = []

for partition in partitions:
    for num_neurons in neurons_list:
        for num_epochs in epochs_list:
            # collect all variants for the group
            candidates = []
            for key, data in indexed_metrics.items():
                p, nprocs, neurons, epochs = key
                if p == partition and neurons == num_neurons and epochs == num_epochs:
                    candidates.append((nprocs, data))

            if not candidates:
                print(f"No data for {partition}, {num_neurons}, {num_epochs}")
                continue

            # find sequential baseline (nprocs = 1)
            seq_key = (partition, 1, num_neurons, num_epochs)
            seq_data = indexed_metrics.get(seq_key)
            if seq_data is None:
                print(f"No sequential baseline for {partition}, {num_neurons}, {num_epochs}")
                continue

            # pick argmin by total time
            best_nprocs, best_data = min(candidates, key=lambda x: x[1]['TOTAL'])

            total = best_data['TOTAL']
            right = best_data['RIGHT']
            speedup = seq_data['TOTAL'] / total
            efficiency = speedup / best_nprocs

            summary_rows.append((partition, num_neurons, num_epochs,
                                 best_nprocs, total, right, speedup, efficiency))

summary_rows.sort(key=lambda x: (x[0], x[1], x[2]))

table = "\\begin{table}[H]\n\\centering\n"
table += "\\begin{tabular}{|c|c|c|c|c|c|c|c|}\n\\hline\n"
table += "Partició & Neurones & Èpoques & Nº Proc. & Temps & Encerts & Speedup & Eficiència \\\\\n\\hline\n"

for (partition, num_neurons, num_epochs,
     nprocs, total, right, speedup, efficiency) in summary_rows:
    table += f"\\texttt{{{partition}}} & {num_neurons} & {num_epochs} & {nprocs} & {total:.4f} & {int(right)} & {speedup:.4f} & {efficiency:.4f} \\\\\n"

table += "\\hline\n\\end{tabular}\n"
table += "\\caption{Millor configuració per temps en cada cas}\n"
table += "\\label{tab:best-configs}\n"
table += "\\end{table}\n"

print(table)

In [ ]:
# SUMMARY TABLE BEST SPEEDUP

partitions = set(k[0] for k in indexed_metrics.keys())
neurons_list = [135, 250]
epochs_list = [10, 100]

summary_speedup = []
summary_eff = []

for partition in partitions:
    for num_neurons in neurons_list:
        for num_epochs in epochs_list:
            # gather all configs in this group
            candidates = []
            for key, data in indexed_metrics.items():
                p, nprocs, neurons, epochs = key
                if p == partition and neurons == num_neurons and epochs == num_epochs:
                    candidates.append((nprocs, data))

            if not candidates:
                print(f"No data for {partition}, {num_neurons}, {num_epochs}")
                continue

            # sequential baseline
            seq_key = (partition, 1, num_neurons, num_epochs)
            seq_data = indexed_metrics.get(seq_key)
            if seq_data is None:
                print(f"No sequential baseline for {partition}, {num_neurons}, {num_epochs}")
                continue

            # compute metrics for all nprocs
            enriched = []
            for nprocs, data in candidates:
                if nprocs == 1: continue
                total = data['TOTAL']
                right = data['RIGHT']
                speedup = seq_data['TOTAL'] / total
                efficiency = speedup / nprocs
                enriched.append((nprocs, total, right, speedup, efficiency))

            # pick max speedup
            best_sp = max(enriched, key=lambda x: x[3])
            summary_speedup.append((partition, num_neurons, num_epochs) + best_sp)

            # pick max efficiency
            best_eff = max(enriched, key=lambda x: x[4])
            summary_eff.append((partition, num_neurons, num_epochs) + best_eff)

# sort for stable LaTeX output
summary_speedup.sort(key=lambda x: (x[0], x[1], x[2]))
summary_eff.sort(key=lambda x: (x[0], x[1], x[2]))

# speedup table
table_sp = "\\begin{table}[H]\n\\centering\n"
table_sp += "\\begin{tabular}{|c|c|c|c|c|c|c|c|}\n\\hline\n"
table_sp += "Partició & Neurones & Èpoques & Nº Proc. & Temps & Encerts & Speedup & Eficiència \\\\\n\\hline\n"
for (partition, num_neurons, num_epochs,
     nprocs, total, right, speedup, efficiency) in summary_speedup:
    table_sp += f"\\texttt{{{partition}}} & {num_neurons} & {num_epochs} & {nprocs} & {total:.4f} & {int(right)} & {speedup:.4f} & {efficiency:.4f} \\\\\n"
table_sp += "\\hline\n\\end{tabular}\n"
table_sp += "\\caption{Millor speedup en cada configuració}\n"
table_sp += "\\label{tab:best-speedup}\n"
table_sp += "\\end{table}\n"

print(table_sp)

In [ ]:
# SUMMARY TABLE BEST EFFICIENCY

table_eff = "\\begin{table}[H]\n\\centering\n"
table_eff += "\\begin{tabular}{|c|c|c|c|c|c|c|c|}\n\\hline\n"
table_eff += "Partició & Neurones & Èpoques & Nº Proc. & Temps & Encerts & Speedup & Eficiència \\\\\n\\hline\n"
for (partition, num_neurons, num_epochs,
     nprocs, total, right, speedup, efficiency) in summary_eff:
    table_eff += f"\\texttt{{{partition}}} & {num_neurons} & {num_epochs} & {nprocs} & {total:.4f} & {int(right)} & {speedup:.4f} & {efficiency:.4f} \\\\\n"
table_eff += "\\hline\n\\end{tabular}\n"
table_eff += "\\caption{Millor eficiència en cada configuració}\n"
table_eff += "\\label{tab:best-efficiency}\n"
table_eff += "\\end{table}\n"

print(table_eff)